# 3.3. Synthetic Regression Data
D2L의 Synthetic Regression Data 장을 PyTorch 기준으로 정리함.

이 장은 선형 회귀용 가짜 데이터를 만드는 것이다.

우리가 정답을 알고있는 데이터를 만들면, 나중에 모델이 그 정답을 제대로 찾아내는지 확인할 수 있기 때문에 가짜 데이터를 사용한다.

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [2]:
%matplotlib inline
import random
import torch
import matplotlib.pyplot as plt
from torch.distributions.multinomial import Multinomial
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

torch.manual_seed(0)
random.seed(0)

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. Synthetic Regression Data가 무엇인가

synthetic data는 직접 만든 가짜 데이터이다.

현실 데이터는 정답 규칙을 잘 모른다. 예를 들어서 집값데이터를 보면, 집값이 정확히 어떻게 결정되는지 모른다. 면적이나 교통, 위치같은것들이 복잡하게 연관된다.

선형회귀를 처음 구현할 때부터 현실 데이터를 사용하면 모델이 안 맞는 이유가 코드 버그때문인지, 데이터가 너무 복잡해서인지, noise가 커서인지 구분하기 어렵기때문이다.

그래서 일부로 쉬운 데이터를 만들어 사용한다.

```py
y = 2 * x1 - 3.4 * x2 + 4.2 + noise # 이렇게 만들면 미리 정답을 알수 있다

true_w = [2, -3.4]
true_b = 4.2

w ≈ [2, -3.4] # 나중에 학습시킬때 이렇게 가까워지면
b ≈ 4.2       # 학습 코드가 정상적으로 작동한다고 판단할 수 있다.
```

## 2. 이 장의 목적

전체 흐름이다.
1. x를 랜덤으로 만든다.
2. 진짜 w, b를 정한다.
3. y = Xw + b + noise 로 label을 만든다.
4. 데이터를 train/validation으로 나눈다.
5. minibatch 단위로 꺼내지도록 DataLoader를 만든다.

여기에서 03_02장에 잠깐 나온 train_dataloader가 어떻게 만들었는지 볼 수 있다.

## 3. 수식 해석

D2L의 핵심 수식이다.
$$
\mathbf{y}= \mathbf{X}\mathbf{w} + b + \boldsymbol{\epsilon}
$$

D2L은 2차원 feature를 가진 예제 1000개를 표준정규분포에서 만들고, lable은 ground-truth linear function에 noise를 더해서 만든다고 설명했다. noise는 평균 0, 표준편차 0.01인 정규분포에서 뽑는다.

- X: 입력 데이터 전체
- w: 진짜 가중치
- b: 진짜 편향
- ε: noise 약간의 랜덤 오차
- y: 정답 label

예를 들어서

```py
x = [x1, x2] # feature가 2개라면 샘플 하나는 이렇게 생겼다.

w = [2, -3.4] # 가중치는 이렇게 정한다.
b = 4.2

y = 2 * x1 + (-3.4) * x2 + 4.2 + noise # 그러면 샘플 하나의 정답은 이렇게 만들어진다.

y = X @ w + b + noise # 이걸 모든 샘플에 한번에 적용하면 이렇게 된다.
```
X @ w는 모든 샘플에 대해 선형식을 한 번 계산하는 행렬곱이다.

## 4. shape으로 이해하기

D2L 코드는 feature가 2개인 데이터를 만든다.

```py
X.shape = (num_examples, 2) # 각 샘플은 숫자 2개를 가진다.

w.shape = (2000, 2) # 예를 들어 train 1000개, validation 1000개를 만들면 전체데이터 개수는 2000개이다.

w.shape = (2,) # w는 feature마다 하나씩 있으므로 길이가 2다.

w.reshape((-1, 1)) # 그런데 행렬곱을 하려면 보통 w를 세로 벡터로 바꿔야 한다.

w.shape = (2, 1) # 그러면 행렬곱은 이렇게 맞는다.

X.shape              # (2000, 2)
w.reshape(-1, 1)     # (2, 1)

X @ w                # (2000, 1)

y.shape = (2000, 1) # 결과적으로 샘플마다 하나의 정답이기 때문에 이렇게 된다.
```

## 5. PyTorch 코드로 직접 구현

D2L 원문은 `d2l.DataModule`을 상속해서 `SyntheticRegressionData` 클래스를 만든다. 핵심 코드는 `X = torch.randn(n, len(w))`, `noise = torch.randn(n, 1) * noise`, `y = X @ w + b + noise`이다.

In [9]:
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

num_examples = 1000
num_features = len(true_w)

X = torch.randn(num_examples, num_features)

noise = torch.randn(num_examples, 1) * 0.01

y = X @ true_w.reshape(-1, 1) + true_b + noise

print("X shape:", X.shape)
print("y shape:", y.shape)

print("first X:", X[0]) # 첫 번째 샘플 feature
print("first y:", y[0]) # 샘플의 정답

X shape: torch.Size([1000, 2])
y shape: torch.Size([1000, 1])
first X: tensor([ 0.9288, -0.0344])
first y: tensor([6.1606])


## 6. noise는 왜 넣는가

noise는 현실 데이터의 불완전한것을 흉내 내는 것이다.
noise가 없으면 데이터가 너무 완벽하게 나올 것이다.

선형회귀 모델은 noise까지 완벽히 외우는게 목적이 아니라 noise가 있더라도 규칙을 찾아야한다.

## 7. train / validation 나누기

D2L에서는 num_train = 1000, num_val = 1000처럼 학습 데이터와 검증 데이터를 따로 둔다.

In [12]:
import torch

true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

num_train = 1000
num_val = 1000
n = num_train + num_val

X = torch.randn(n, len(true_w))
noise = torch.randn(n, 1) * 0.01
y = X @ true_w.reshape(-1, 1) + true_b + noise

X_train = X[:num_train]
y_train = y[:num_train]

X_val = X[num_train:]
y_val = y[num_train:]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

X_train: torch.Size([1000, 2])
y_train: torch.Size([1000, 1])
X_val: torch.Size([1000, 2])
y_val: torch.Size([1000, 1])


## 8. 왜 w.reshape((-1, 1))을 쓰나

In [ ]:
# 처음 w는 이렇다
w = torch.tensor([2.0, -3.4])
print("w.shape: ", w.shape)

# 근데 이건 (2,) shape다. 1차원 벡터이다. 근데 X는 2차원 행렬이다.

# 결과y를 (1000, 1) 모양으로 만들고 싶다. 그래서 w를 세로 벡터로 바꾼다.
print("w.reshape((-1, 1)) :" , w.reshape((-1, 1)))

# 이렇게 하면 행렬곱이 명확해진다.

w.shape:  torch.Size([2])
w.reshape((-1, 1)) : tensor([[ 2.0000],
        [-3.4000]])


## 9. 데이터 하나 확인하기

D2L은 첫 번째 feature와 label을 출력해서 데이터가 어떻게 생겼는지 확인한다. 원문에서도 각 row의 feature는 2차원 벡터이고, label은 scalar라고 설명한다.

## 10. minibatch가 무엇인가

전체 데이터를 1000개라고 하고, 학습할 때 1000개를 전부 넣을 수도 있지만, 보통은 작은 묶음으로 나눠 넣는다. 이 묶음을 minibatch이라고 한다.

예를 들면
```py
batch_size = 32 # 한 번에 32개 샘플씩 꺼낸다.

X_batch.shape = (32, 2)
y_batch.shape = (32, 1)

X shape: torch.Size([32, 2])
y shape: torch.Size([32, 1])
```

## 11. 직접 minibatch를 만드는 방식

D2L은 먼저 직접 index를 섞고, batch단위로 잘라 데이터를 꺼내는 방식을 보여준다.
학습 모드에서는 데이터를 radnom order로 읽고, validation 모드에서는 정해진 순서로 읽을 수 있다고 한다.

In [ ]:
import random # 순수 PyTorch
import torch

batch_size = 32

indices = list(range(num_train))
random.shuffle(indices) # 학습 데이터 섞기

for i in range(0, num_train, batch_size):
    batch_indices = torch.tensor(indices[i:i + batch_size])

    X_batch = X_train[batch_indices]
    y_batch = y_train[batch_indices]

    print(X_batch.shape)
    print(y_batch.shape)

    break

torch.Size([32, 2])
torch.Size([32, 1])


섞는 이유는 모델이 데이터 순서에 이상하게 영향을 받지 않게 하려고 섞는다.  
예를 들어서 데이터가 작은 값부터 큰 값 순서로 정렬되어 있으면, batch마다 데이터 분포가 달라질 수 있다.  
그러면 학습이 불안정해질 수 있다. 검증은 학습이 아니라 평가이기 때문에 보통 섞지 않아도 된다.

## 12. 직접 minibatch를 만들 필요는 없다.

실제로는 PyTorch의 DataLoader를 쓴다고 한다.
D2L도 직접 만든 iterator는 모든 데이터를 메모리에 올려야하고 random memory access가 많아서 실제 문제에서는 비효율적일 수 있다고 설명한다.  
그래서 framework의 built-in iterator를 쓰는 방법을 소개한다.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(X_train, y_train) # TensorDataset은 x와 y를 하나로 묶는다.
val_dataset = TensorDataset(X_val, y_val)

batch_size = 32

train_loader = DataLoader( # DataLoader는 dataset에서 batch단위로 꺼내준다.
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

## 12. DataLoader 확인하기

In [16]:
X_batch, y_batch = next(iter(train_loader))

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

len(train_loader)

X_batch shape: torch.Size([32, 2])
y_batch shape: torch.Size([32, 1])


32

In [19]:
import torch
from torch.utils.data import TensorDataset, DataLoader

# 1. 재현성을 위해 random seed 고정
torch.manual_seed(42)

# 2. 진짜 정답 파라미터
true_w = torch.tensor([2.0, -3.4])
true_b = 4.2

# 3. 데이터 개수
num_train = 1000
num_val = 1000
n = num_train + num_val

# 4. feature 생성
X = torch.randn(n, len(true_w))

# 5. noise 생성
noise = torch.randn(n, 1) * 0.01

# 6. label 생성
y = X @ true_w.reshape(-1, 1) + true_b + noise

# 7. train / validation 분리
X_train = X[:num_train]
y_train = y[:num_train]

X_val = X[num_train:]
y_val = y[num_train:]

# 8. TensorDataset 생성
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

# 9. DataLoader 생성
batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

# 10. batch 확인
X_batch, y_batch = next(iter(train_loader))

print("X shape:", X.shape)
print("y shape:", y.shape)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_batch shape:", X_batch.shape)
print("y_batch shape:", y_batch.shape)

print("first X_batch:", X_batch[0])
print("first y_batch:", y_batch[0])

X shape: torch.Size([2000, 2])
y shape: torch.Size([2000, 1])
X_train shape: torch.Size([1000, 2])
y_train shape: torch.Size([1000, 1])
X_batch shape: torch.Size([32, 2])
y_batch shape: torch.Size([32, 1])
first X_batch: tensor([-0.3123, -0.2542])
first y_batch: tensor([4.4262])


## 13. 오늘의 정리

- Synthetic data는 직접 만든 가짜 데이터다.
- 이 장에서는 선형회귀 학습을 테스트하기 위한 데이터를 만든다.
- 핵심 수식은 y = Xw + b + noise 이다.
- X는 입력 feature matrix다.
- y는 정답 label이다.
- w는 feature별 가중치다.
- b는 편향이다.
- noise는 현실 데이터의 오차를 흉내 내는 작은 랜덤값이다.
- X.shape = (1000, 2)는 샘플 1000개, feature 2개라는 뜻이다.
- y.shape = (1000, 1)는 샘플마다 label 1개가 있다는 뜻이다.
- w.reshape(-1, 1)은 행렬곱을 위해 w를 세로 벡터로 바꾸는 것이다.
- batch_size=32는 한 번에 샘플 32개씩 꺼낸다는 뜻이다.
- TensorDataset은 X와 y를 묶어준다.
- DataLoader는 dataset에서 batch를 꺼내준다.
- train_loader는 보통 shuffle=True를 쓴다.
- validation loader는 보통 shuffle=False를 쓴다.
- D2L의 DataModule, add_to_class, save_hyperparameters는 지금 단계에서 핵심이 아니다.
- 이 장의 핵심은 순수 PyTorch로 X, y, DataLoader를 만들 수 있는가이다.